# AI Agent 기반 농업 정보 검색 실습 답안

농업기술길잡이 **PDF RAG**, 노지·스마트팜 **로컬 CSV**, API 키 없는 **웹 검색**을
LangChain Tool로 연결한 농업 정보 검색 Agent입니다.

```text
                 AI 농업 Agent
                      │
       ┌──────────────┼──────────────┐
       ↓              ↓              ↓
   농업기술 RAG    농업 CSV        웹 검색
       │              │              │
       ↓              ↓              ↓
   PDF 자료      실제 데이터     최신 정보
       └──────────────┼──────────────┘
                      ↓
                    LLM
                      ↓
              농업 정보 답변
```

- OpenAI 키: `C:\env\.env` 의 `OPENAI_API_KEY` 만 사용
- 기상청·농업기술 데이터 플랫폼·Tavily 등 다른 API는 사용하지 않음
- 숫자는 CSV, 재배법은 PDF, 최신 이슈는 웹 검색 결과만 근거로 답함


## STEP 0. 환경 준비

`C:\env\.env`에서 `OPENAI_API_KEY`만 읽고 값은 출력하지 않습니다.


In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain.agents import create_agent

from agri_search_core import (
    FIELD_CROPS,
    PDF_CROPS,
    SAMPLE_QUESTIONS,
    SMART_CROPS,
    TOOL_LABELS,
    ask_agent,
    build_runtime,
    ensure_extracted,
    inventory_tables,
    load_keys,
    sample_heads,
)

pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 90)

keys = load_keys()
print("OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("create_agent:", create_agent.__module__)
print("LLM: gpt-4o-mini / Embedding: text-embedding-3-small")
print("PDF 작물:", ", ".join(PDF_CROPS))
print("노지 CSV:", ", ".join(FIELD_CROPS))
print("스마트팜 CSV:", ", ".join(SMART_CROPS))


OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)
create_agent: langchain.agents.factory
LLM: gpt-4o-mini / Embedding: text-embedding-3-small
PDF 작물: 토마토, 고추, 수박, 참외, 고구마, 딸기, 사과
노지 CSV: 고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도
스마트팜 CSV: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카


## STEP 1. 자료 해제와 목록 파악

`노지_2024.zip`, `스마트팜_2024.zip`을 `data/`에 풀고 파일·컬럼·작물을 표로 남깁니다.
환경 CSV는 대용량이므로 head만 확인하고 전 행을 출력하지 않습니다.


In [2]:
roots = ensure_extracted()
print("노지 해제:", roots["field"])
print("스마트팜 해제:", roots["smart"])

inventory = inventory_tables()
display(inventory[["구분", "파일", "행 수", "주요 컬럼", "작물", "기간 힌트"]])

print("\n샘플 1~2행 (식별자·키 없음)")
for title, df in sample_heads().items():
    print(f"\n[{title}]")
    display(df)


노지 해제: C:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\04_AI Agent 기반 농업 정보 검색 서비스 개발 실습\data\노지_2024
스마트팜 해제: C:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\04_AI Agent 기반 농업 정보 검색 서비스 개발 실습\data\스마트팜_2024


,구분,파일,행 수,주요 컬럼,작물,기간 힌트
0,노지 농가,공개용_2024_농가정보.csv,312,"연도, 지역(도), 시군, 농가명, 작목, 품종, 포장면적, 주간거리","고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도",2023-09-10 ~ 2024-10-17
1,노지 생육,생육기본_고추_2024.csv,5252,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 초장, 착과수",고추,2024-05-17 ~ 2024-05-17
2,노지 생육,생육기본_마늘_2024.csv,13809,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 개체번호, 엽수",마늘,2024-04-02 ~ 2024-04-02
3,노지 생육,생육기본_밀_2024.csv,4235,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 초장, 수수",밀,2024-02-06 ~ 2024-02-06
4,노지 생육,생육기본_배추_2024.csv,2733,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 초장(엽장), 엽수",배추,2024-09-13 ~ 2024-09-13
5,노지 생육,생육기본_사과_2024.csv,8089,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 수고, 수폭",사과,2024-06-07 ~ 2024-06-07
6,노지 생육,생육기본_양파_2024.csv,9223,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 개체번호, 엽수",양파,2024-01-10 ~ 2024-01-10
7,노지 생육,생육기본_옥수수_2024.csv,945,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 개체번호, 출웅기",옥수수,2024-06-04 ~ 2024-06-04
8,노지 생육,생육기본_콩_2024.csv,13399,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 조사번호, 초장",콩,2024-06-27 ~ 2024-06-27
9,노지 생육,생육기본_포도_2024.csv,4058,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 주간굵기, 주간높이",포도,2024-06-21 ~ 2024-06-21



샘플 1~2행 (식별자·키 없음)

[노지 농가 샘플]


,연도,지역(도),시군,농가명,작목,품종,포장면적,주간거리,조간거리,파종일자,정식일자,수확일자,총수확량,비고
0,2024,경기,안성,1,고추,뚝심칼탄,660,0.26,NaN,NaN,2024-04-29,2024-10-02,336,NaN
1,2024,경기,화성,2,고추,칼탄맥스,1155,0.45,NaN,NaN,2024-05-01,2024-09-03,139,NaN



[노지 고추 생육 샘플]


,시도,시군구,품목,농가명,조사일,개체번호,초장,착과수,수확과수,비고
0,경기,안성,고추,1,2024-05-17,1,30.5,0,0,NaN
1,경기,안성,고추,1,2024-05-17,2,37.1,0,0,NaN



[스마트팜 환경 head]


,도,시군,품목,작기,농가명,측정시간,온도_외부,풍향_외부,...,일사량_외부,누적일사량_외부,강우감지,온도_내부,상대습도_내부,잔존CO2,토양온도,Unnamed: 16
0,충북,산청,딸기,1,1,2024-10-19 9:02,NaN,NaN,...,0,NaN,NaN,20.1,91.7,NaN,NaN,NaN
1,충북,산청,딸기,1,1,2024-10-19 10:02,NaN,NaN,...,0,NaN,NaN,20.9,89.7,NaN,NaN,NaN


## STEP 2~5. RAG · CSV · 웹 검색 Tool + Agent

PDF 7종을 FAISS에 persist하고 (`C:\env\agri_search_faiss`),
노지/스마트팜 CSV 요약 Tool과 DuckDuckGo 웹 검색 Tool을 장착합니다.


In [3]:
runtime = build_runtime()
tools = runtime["tools"]
print("PDF 파일 수:", runtime["pdf_count"])
print("벡터 청크 수:", runtime["vector_count"])
print("인덱스 persist:", runtime["index_ready"])
print("장착 Tool:", list(tools))


PDF 파일 수: 7
벡터 청크 수: 2160
인덱스 persist: True
장착 Tool: ['search_crop_guide', 'query_field_csv', 'query_smartfarm_csv', 'search_web']


In [4]:
print("[STEP 2] search_crop_guide 미리보기 (고구마 정식)")
print(tools["search_crop_guide"].invoke({"query": "정식과 물 관리", "crop": "고구마"})[:1200])
print("\n[STEP 3] query_field_csv 미리보기 (노지 고추 생육)")
print(tools["query_field_csv"].invoke({"crop": "고추", "kind": "growth"})[:1200])
print("\n[STEP 3] query_smartfarm_csv 미리보기 (완숙토마토 환경)")
print(
    tools["query_smartfarm_csv"].invoke(
        {"crop": "완숙토마토", "kind": "environment"}
    )[:1200]
)
print("\n[STEP 4] search_web 미리보기")
print(tools["search_web"].invoke({"query": "딸기 병해충 최근 이슈 농촌진흥청"})[:1200])


[STEP 2] search_crop_guide 미리보기 (고구마 정식)


아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.

[1] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=109
109제Ⅳ장 재배법 묘상관리라 (1) 싹이 트기까지의 관리 씨고구마를 묻은 다음에는 적당한 온도 유지와 물주기에 주의를 하 여야 한다. 상토의 온도가 38℃ 이상이면 고구마가 썩을 위험이 있고 상 토 위에 피복물을 덮은 경우 35℃를 넘으면 좋지 않다. 씨고구마를 묻은 후 싹이 트기까지 묘상에서는 가급적 고온인 30～ 33℃를 유지시켜야 싹이 고르게 빨리 나온다. 온도는 온도계를 묘상의  여러 곳에 꽂아 조사하면서 관리한다. 이때 너무 깊이 꽂으면 양열온상 의 경우 발열재료에 닿거나 전열온상의 전열선에 닿으면 온도가 상토  온도보다 높게 나타나므로 고구마가 묻힌 부분에 온도계의 끝이 머무르 도록 한다.  물은 씨고구마를 묻은 후 충분히 주어 씨고구마가 마르지 않도록 하 고 이후에도 상토표면이 마르지 않도록 한다. 냉수를 주기보다는 미리  물통에 담아 두었다가 냉기가 가신 다음에 주도록 한다. 싹이 트기까지  걸리는 기간은 묘상의 종류나 상태에 따라서 다르나 온상에서는 7～10 일, 비닐냉상에서는 2～3주일 정도 걸린다. 씨고구마를 묻은 후 온도와 습도 관리가 잘되지 않을 경우에는 병 발 생률이 높아진다. 고구마의 모든 품종에 일치할 수는 없으나 대표적인  점질고구마와 분질고구마의 일부 품종을 대상으로 연구한 결과 평균 온 도가 35℃ 이상 및 습도 30% 이상으로 유지되면 병 발생률이 높아지며  맹아율도 품종에 따라 현저히 떨어지는 ...

[2] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=163
163제Ⅴ장 수확 및 저장 고구마는 캐면 호흡이 왕성해진다. 호흡작용은 수확 후 급격히 증가하 여 7～10일간 높고 그 후 차츰 낮아져 15～20일 후에는 안정하게 된다. 수 확한 다음 바로 고구마를 쌓아 두면 호흡과 수분 발산이 왕성하여 온도가  높아지고, 습해지면 탄산가스의 농도가 높아서 시간이 

{
  "자료": "스마트팜_2024 환경 (필터·요약, 원문 전체 아님)",
  "작물": [
    "완숙토마토"
  ],
  "건수": 130910,
  "기간": "2024-01-07 0:00 ~ 2025-07-15 9:00",
  "지역": "시도=전북:32407, 전남:32269, 강원:26272, 경기:21384, 경남:18578 / 시군=광주:21384, 화순:16093, 평창:15513, 김제:14815, 사천:14380, 철원:10759, 익산:10488, 보성:9290",
  "통계": {
    "온도_내부": {
      "건수": 130762,
      "평균": 20.37,
      "최소": 0.0,
      "최대": 55.8
    },
    "상대습도_내부": {
      "건수": 125753,
      "평균": 83.27,
      "최소": 12.2,
      "최대": 100.0
    },
    "잔존CO2": {
      "건수": 115073,
      "평균": 557.07,
      "최소": 1.0,
      "최대": 5118.5
    },
    "토양온도": {
      "건수": 54343,
      "평균": 20.89,
      "최소": 6.9,
      "최대": 44.9
    },
    "일사량_외부": {
      "건수": 95897,
      "평균": 158.38,
      "최소": 0.0,
      "최대": 1397.0
    },
    "온도_외부": {
      "건수": 104980,
      "평균": 13.16,
      "최소": -15.4,
      "최대": 40.3
    }
  },
  "안내": "환경 CSV는 대용량이므로 품목 필터 후 평균·최소·최대만 반환했습니다."
}

[STEP 4] search_web 미리보기


웹 검색은 최신 이슈 보강용입니다. 재배 매뉴얼·실측 숫자로 쓰지 마세요.



[1] 제목: 경남농업기술원, 농촌진흥청장과 경남 농업현안 공유 < 농촌지방 ...
URL: https://www.rda.go.kr/board/board.do?prgId=day_farmlcltinfoEntry&dataNo=100000812683&mode=updateCnt
발췌: - 27일, 최고농업기술명인 심포지엄 참석, 기후변화 대응 농업기술 확산 논의 - 전국 딸기 주산지 경남, 종자번식 딸기 품종 개발 추진현황 공유 - 고성 벼 병해충 예찰·방제 현장 찾아 신속 대응체계 점검 경상남도농업기술원(원장 정찬식)은 27일 농촌진흥청장이 경남농업기술원을 방문해 주요 연구 ...

[2] 제목: 농촌진흥청장, 경남 딸기 연구·벼 병해충 대응 현장 점검농촌 ...
URL: https://www.newstown.co.kr/news/articleView.html?idxno=713889
발췌: 경상남도농업기술원은 27일 농촌진흥청장이 기술원을 방문해 주요 연구시설과 영농현장을 둘러보고 병해충 예찰·방제 대응 상황을 점검했다고 밝혔다. 이번 방문은 기후변화와 병해충 발생 증가 등 급변하는 농업환경에 대응해 중앙과 지방 농촌진흥기관의 협력을 강화하고 경남 농업현장의 주요 ...

[3] 제목: 국가농작물병해충관리시스템
URL: https://ncpms.rda.go.kr/npms/Main.np
발췌: 국가농작물병해충관리시스템 신속한 조기경보와 대응으로 농작물 피해를 줄입니다. (오이 - 오이) 오이잎이 이상해요. 주간농사정보 제 35호 (2026.8.31.~9.6.) 주간농사정보 제 34호 (2026.8.24.~8.30.) 주간농사정보 제 33호 (2026.8.17.~8.23.) 주간농사정보 제 32호 (2026.8.10.~8.16.)

[4] 제목: 봄철 딸기 품질 지키기 '환경·양분·병해충' 관리 요령 제시
URL: https://periai.kr/ko/gov-press/156746729-봄철

## STEP 6. 실행 시나리오 A~E

같은 Agent로 다섯 질문을 실행하고, 호출된 Tool과 최종 답을 남깁니다.


In [5]:
results = []
for item in SAMPLE_QUESTIONS:
    print("=" * 72)
    print(item["label"])
    print(item["text"])
    out = ask_agent(runtime, item["text"])
    results.append({**item, **out})
    called = out["tools"]
    labels = [TOOL_LABELS.get(name, name) for name in called]
    print("호출 Tool:", labels if labels else "(없음)")
    print("\n[트레이스]")
    for tr in out["trace"]:
        if tr.get("label") == "결과":
            print(f"  결과 {tr.get('name')}: {tr.get('preview', '')[:160]}")
        else:
            print(f"  호출 {tr.get('label')}: {tr.get('args')}")
    print("\n[최종 답변]")
    print(out["answer"])
    print()


A. PDF만
고구마 정식과 물 관리 요령을 농업기술길잡이 기준으로 요약해줘.


호출 Tool: ['농업기술 RAG', '농업기술 RAG']

[트레이스]
  호출 농업기술 RAG: {'query': '정식', 'crop': '고구마'}
  호출 농업기술 RAG: {'query': '관수', 'crop': '고구마'}
  결과 search_crop_guide: 아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.  [1] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=20 20 농업기술길잡이 고구마  2016년 국내 고구마 생산량은 34만 1,000톤으로 전남(광주 포함)  24.9%, 전북 18%, 경기(서울
  결과 search_crop_guide: 아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.  [1] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=205 205제Ⅵ장 병해충 방제 및 생리장해 고구마는 환경적응성 및 광 보상력이 뛰어나 식량 작물 중 단위면적당  수량성이 크며 다른 작물에 비하

[최종 답변]
고구마의 정식과 물 관리 요령은 다음과 같습니다.

### 정식 요령
- 고구마는 씨고구마를 준비하여 심습니다. 씨고구마는 품종이 확실하고, 영농규모와 재배 시기를 조절할 수 있는 것이 좋습니다.
- 재배적지는 교통이 편리하고, 인력작업이 많아 집단재배가 이루어져야 하며, 비옥하고 배수가 양호한 토양에서 재배해야 합니다.
- 관수는 자주 해야 하며, 점적관수나 스프링클러 등 자동화된 관수 시스템을 사용하는 것이 바람직합니다. (출처: 농업기술길잡이 고구마, 페이지 146)

### 물 관리 요령
- 고구마는 생육 기간 동안 충분한 수분을 공급해야 하며, 특히 초기 생육 단계에서 물 부족이 발생하지 않도록 주의해야 합니다.
- 관수 방법으로는 점적관수와 스프링클러를 활용하여 수분을 고르게 공급하는 것이 좋습니다. (출처: 농업기술길잡이 고구마, 페이지 205)

이 정보는 농업기술길잡이 고구마 PDF에서 발췌한 내용입니다.

B. 노지 CSV만
2024년 노지 

호출 Tool: ['노지 CSV']

[트레이스]
  호출 노지 CSV: {'crop': '고추', 'kind': 'growth'}
  결과 query_field_csv: {   "자료": "노지_2024 생육기본 (생육기본_고추_2024.csv)",   "작물": "고추",   "건수": 5252,   "기간": "2024-05-17 ~ 2024-10-14",   "지역": "시도=경북:3100, 경기:1000, 전남:612, 충남:540 / 시군=안동

[최종 답변]
2024년 노지 고추 생육에 대한 요약은 다음과 같습니다.

- **생육 기간**: 2024-05-17 ~ 2024-10-14
- **총 조사 건수**: 5,252건

### 생육 통계
- **초장**:
  - 평균: 88.54 cm
  - 최소: 17.0 cm
  - 최대: 179.0 cm
- **착과수**:
  - 평균: 37.1개
  - 최소: 0.0개
  - 최대: 285.0개

### 지역별 생육 정보
1. **경기**
   - 조사 건수: 1,000
   - 초장 평균: 87.43 cm
   - 착과수 평균: 31.82개

2. **경북**
   - 조사 건수: 3,100
   - 초장 평균: 80.7 cm
   - 착과수 평균: 34.02개

3. **전남**
   - 조사 건수: 612
   - 초장 평균: 112.14 cm
   - 착과수 평균: 59.98개

4. **충남**
   - 조사 건수: 540
   - 초장 평균: 106.69 cm
   - 착과수 평균: 36.84개

이 자료는 "노지_2024 생육기본 (생육기본_고추_2024.csv)"에서 발췌한 것입니다.

C. 스마트팜 CSV만
스마트팜 완숙토마토 내부 온습도가 어떤지, 출하량도 같이 요약해줘.


호출 Tool: ['스마트팜 CSV', '스마트팜 CSV']

[트레이스]
  호출 스마트팜 CSV: {'crop': '완숙토마토', 'kind': 'environment'}
  호출 스마트팜 CSV: {'crop': '완숙토마토', 'kind': 'production'}
  결과 query_smartfarm_csv: {   "자료": "스마트팜_2024 환경 (필터·요약, 원문 전체 아님)",   "작물": [     "완숙토마토"   ],   "건수": 130910,   "기간": "2024-01-07 0:00 ~ 2025-07-15 9:00",   "지역": "시도=전북:32407, 전남:322
  결과 query_smartfarm_csv: {   "자료": "스마트팜_2024 생산",   "작물": [     "완숙토마토"   ],   "건수": 6531,   "기간": "2024-03-25 ~ 2025-08-04",   "지역": "시도=경남:3929, 강원:1110, 전북:1098, 전남:339, 경기:55 / 시군=

[최종 답변]
스마트팜에서의 완숙토마토 내부 온습도와 출하량에 대한 요약은 다음과 같습니다.

### 내부 온습도
- **온도**: 
  - 평균: 20.37°C
  - 최소: 0.0°C
  - 최대: 55.8°C
- **상대습도**: 
  - 평균: 83.27%
  - 최소: 12.2%
  - 최대: 100.0%

### 출하량
- **총 출하량**: 
  - 평균: 400.41 kg
  - 최소: 1.0 kg
  - 최대: 22,000.0 kg
- **판매 금액**: 
  - 평균: 1,236,262.06 원
  - 최소: 191.0 원
  - 최대: 79,328,000.0 원

이 데이터는 2024년 1월 7일부터 2025년 7월 15일까지의 환경 데이터와 2024년 3월 25일부터 2025년 8월 4일까지의 출하량 데이터를 기반으로 합니다.

D. RAG + CSV + 웹
딸기 재배 주의사항을 길잡이에서 찾고, 스마트팜 딸기 

호출 Tool: ['농업기술 RAG', '스마트팜 CSV', '스마트팜 CSV', '웹 검색']

[트레이스]
  호출 농업기술 RAG: {'query': '재배 주의사항', 'crop': '딸기'}
  호출 스마트팜 CSV: {'crop': '딸기', 'kind': 'growth'}
  호출 스마트팜 CSV: {'crop': '딸기', 'kind': 'environment'}
  호출 웹 검색: {'query': '딸기 최근 이슈'}
  결과 search_crop_guide: 아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.  [1] 작물=딸기 | 파일=농업기술길잡이40_딸기.PDF | 페이지=117 딸기 / 121 (표 35) 딸기 시설 재배 기간별 주(週) 단위 관비 공급량(농과원, 2018) 딸기 재배기간(9월∼5월) 생육 단계(wee
  결과 query_smartfarm_csv: {   "자료": "스마트팜_2024 생육",   "작물": [     "딸기"   ],   "요약": [     {       "작물": "딸기",       "파일": "생육_딸기_2024.csv",       "건수": 14100,       "기간": "2024-09-23 ~ 2
  결과 query_smartfarm_csv: {   "자료": "스마트팜_2024 환경 (필터·요약, 원문 전체 아님)",   "작물": [     "딸기"   ],   "건수": 133718,   "기간": "2024-09-01 0:00 ~ ㄴ",   "지역": "시도=전남:47632, 충남:34344, 전북:18960, 경남:
  결과 search_web: 웹 검색은 최신 이슈 보강용입니다. 재배 매뉴얼·실측 숫자로 쓰지 마세요.    [1] 제목: "딸기값 40% 폭등했는데"…농가는 왜 딸기를 버릴까? | 세계일보 URL: https://www.segye.com/newsView/20260105501307 발췌: 비싸진 딸기 vs. 웃지 

[최종 답변]
### 딸기 재

호출 Tool: ['노지 CSV', '스마트팜 CSV']

[트레이스]
  호출 노지 CSV: {'crop': '딸기', 'kind': 'growth'}
  호출 스마트팜 CSV: {'crop': '사과', 'kind': 'environment'}
  결과 query_field_csv: 노지_2024 CSV에 '딸기' 자료가 없습니다. 가능한 작물: 고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도. 다른 작물 숫자로 대체하지 마세요. 주어진 자료만으로는 알 수 없다고 답하세요.
  결과 query_smartfarm_csv: 스마트팜_2024 CSV에 '사과' 자료가 없습니다. 가능한 작물: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카. 다른 작물 숫자로 대체하지 마세요. 주어진 자료만으로는 알 수 없다고 답하세요.

[최종 답변]
주어진 자료만으로는 알 수 없습니다.



## STEP 7. 결과 정리


In [6]:
def has_pdf(answer: str, tools: list[str]) -> str:
    return "O" if "search_crop_guide" in tools and ("페이지" in answer or "파일" in answer or "길잡이" in answer) else "X"

def has_csv(answer: str, tools: list[str]) -> str:
    csv_hit = "query_field_csv" in tools or "query_smartfarm_csv" in tools
    return "O" if csv_hit else "X"

def has_web(answer: str, tools: list[str]) -> str:
    return "O" if "search_web" in tools else "X"

rows = []
for item in results:
    rows.append(
        {
            "시나리오": item["id"],
            "질문 의도": item["label"],
            "기대한 Tool": ", ".join(item["expected"]),
            "실제 호출 Tool": ", ".join(item["tools"]) or "(없음)",
            "PDF 근거": has_pdf(item["answer"], item["tools"]),
            "CSV 근거": has_csv(item["answer"], item["tools"]),
            "웹 근거": has_web(item["answer"], item["tools"]),
            "비고": item["answer"][:80].replace("\n", " "),
        }
    )
summary = pd.DataFrame(rows)
display(summary)

d = next(item for item in results if item["id"] == "D")
display(Markdown("### 시나리오 D 최종 답변 전문"))
display(Markdown(d["answer"]))


,시나리오,질문 의도,기대한 Tool,실제 호출 Tool,PDF 근거,CSV 근거,웹 근거,비고
0,A,A. PDF만,search_crop_guide,"search_crop_guide, search_crop_guide",O,X,X,"고구마의 정식과 물 관리 요령은 다음과 같습니다. ### 정식 요령 - 고구마는 씨고구마를 준비하여 심습니다. 씨고구마는 품종이 확실하고, 영"
1,B,B. 노지 CSV만,query_field_csv,query_field_csv,X,O,X,2024년 노지 고추 생육에 대한 요약은 다음과 같습니다. - **생육 기간**: 2024-05-17 ~ 2024-10-14 - **총 조사
2,C,C. 스마트팜 CSV만,query_smartfarm_csv,"query_smartfarm_csv, query_smartfarm_csv",X,O,X,스마트팜에서의 완숙토마토 내부 온습도와 출하량에 대한 요약은 다음과 같습니다. ### 내부 온습도 - **온도**: - 평균: 20.37
3,D,D. RAG + CSV + 웹,"search_crop_guide, query_smartfarm_csv, search_web","search_crop_guide, query_smartfarm_csv, query_smartfarm_csv, search_web",O,O,O,### 딸기 재배 주의사항 딸기 재배 시 주의해야 할 사항은 다음과 같습니다: 1. **수분 관리**: 딸기는 전 생육기간 동안 수분 관리가
4,E,E. 자료 없음,자료 없음 명시,"query_field_csv, query_smartfarm_csv",X,O,X,주어진 자료만으로는 알 수 없습니다.


### 시나리오 D 최종 답변 전문

### 딸기 재배 주의사항
딸기 재배 시 주의해야 할 사항은 다음과 같습니다:

1. **수분 관리**: 딸기는 전 생육기간 동안 수분 관리가 매우 중요합니다. 1포기당 흡수량은 약 27L이며, 관수량과 관수 횟수는 토양의 수분 보유력에 따라 조절해야 합니다. 점적관수시설을 사용할 경우, 정식 후부터 적절한 수분을 공급해야 합니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 117)

2. **재배 작형 선택**: 최근에는 촉성재배가 주류를 이루고 있으며, 품종 선택 시 휴면성과 수확기를 고려해야 합니다. 촉성재배용 품종은 화아분화가 잘 이루어져야 하며, 연속 출뢰성이 중요합니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 104)

3. **토양 소독**: 연작장해를 방지하기 위해 효과적인 토양 소독이 필요합니다. 여름철 고온기에 유기물을 투입하여 태양열 소독을 실시하는 것이 좋습니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 111)

4. **수확 요령**: 수확 시 물리적 손상을 최소화하고, 수확한 과실은 적절한 온도에서 관리해야 합니다. 수확 시간은 기온을 고려하여 결정해야 합니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 150)

### 스마트팜 딸기 생육 및 환경 데이터
- **생육 데이터**:
  - **건수**: 14,100
  - **기간**: 2024-09-23 ~ 2025-06-18
  - **주요 생육 통계**:
    - 초장: 평균 35.57 cm (최소 8.0 cm, 최대 64.5 cm)
    - 엽장: 평균 8.69 cm (최소 2.7 cm, 최대 17.4 cm)
    - 엽폭: 평균 7.01 cm (최소 1.9 cm, 최대 14.7 cm)
    - 엽수: 평균 12.94 (최소 2.0, 최대 56.0)
    - 화방별 착과수: 평균 4.09 (최소 1.0, 최대 27.0)

- **환경 데이터**:
  - **건수**: 133,718
  - **주요 환경 통계**:
    - 내부 온도: 평균 16.24°C (최소 0.0°C, 최대 48.8°C)
    - 상대 습도: 평균 84.19% (최소 7.9%, 최대 100.0%)
    - 잔존 CO2: 평균 510.46 ppm (최소 13.6 ppm, 최대 4078.0 ppm)
    - 토양 온도: 평균 20.04°C (최소 1.0°C, 최대 63.5°C)

### 최근 딸기 관련 이슈
1. **딸기 가격 상승**: 딸기 가격이 연초부터 40% 급등했으나, 농가는 수익이 낮아 어려움을 겪고 있습니다. (출처: [세계일보](https://www.segye.com/newsView/20260105501307))

2. **수확 후 폐기 문제**: 높은 가격에도 불구하고, 수확한 딸기가 제값을 받지 못해 폐기되는 사례가 발생하고 있습니다. (출처: [조선일보](https://www.chosun.com/national/national_general/2026/01/06/U4LFCYGM55H5TGQN4ZA62EYN4U/))

3. **신품종 개발**: 여름딸기 신품종 '예랑'이 다수확을 기록하며 주목받고 있습니다. (출처: [오토트리뷴](https://www.autotribune.co.kr/news/articleView.html?idxno=44384))

4. **K-딸기 인기**: 한국 딸기가 해외 시장에서 인기를 끌고 있으며, 신품종 개발로 다양한 맛과 크기가 제공되고 있습니다. (출처: [KBS 뉴스](https://news.kbs.co.kr/news/pc/view/view.do?ncd=8148416))